# Pipeline đầy đủ — 4 phase trong 1 file (có toggle)

Bật/tắt từng phase ở cell **Config**. Chạy theo thứ tự Phase 1 → 0 → 2 → 3 trong **một**
session, nên checkpoint của Phase 1 (gồm `cmo`) tự có cho Phase 0/3.

- **CIFAR-100-LT:** bật cả 4.
- **CUB-200-LT:** đặt `DATA_DIR=.../CUB-200-LT`, `MANY/FEW_THRESHOLD=15/6`; track from-scratch
  (Phase 1) cần `IMAGE_SIZE>=64` và yếu — có thể tắt `RUN_PHASE1/RUN_PHASE0`, đặt `USE_CMO=False`.
- **2× T4:** `USE_MULTI_GPU=True` chia batch trích feature cho cả 2 GPU (an toàn, forward-only).

## 0. Setup (open_clip + transformers nếu chạy track CLIP)

In [ ]:
import importlib.util, sys, subprocess
for pkg, mod in [("open_clip_torch", "open_clip"), ("transformers", "transformers")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("deps ready")

## 1. Project root

In [ ]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if base.exists():
            candidates += [p for p in base.iterdir() if p.is_dir()]
    for path in candidates:
        if (path / "src" / "__init__.py").exists():
            return path
    return Path.cwd()


PROJECT_DIR = find_project_root()
sys.path.insert(0, str(PROJECT_DIR))
print("Project root:", PROJECT_DIR)

## 2. Config (chỉ cần sửa cell này)

In [ ]:
from src.datasets.cifar_lt import load_class_counts

# ===== phase nào chạy =====
RUN_PHASE1 = True    # from-scratch (run_all)            — nặng, CIFAR 32px
RUN_PHASE0 = True    # reuse: ensemble / tier / tau / CLIP-fusion (cần checkpoint Phase 1)
RUN_PHASE2 = True    # adapt CLIP: Tip-Adapter / LIFT / FT-ablation
RUN_PHASE3 = True    # research: LLM / DINOv2 / diffusion / mixup / GLA / fusion
USE_MULTI_GPU = True # chia batch trích feature cho mọi GPU (Kaggle 2x T4)

# ===== dataset =====
DATA_DIR = PROJECT_DIR / "data" / "CIFAR-100-LT"   # hoặc .../CUB-200-LT
OUTPUT_DIR = PROJECT_DIR / "outputs"
NUM_CLASSES = len(load_class_counts(DATA_DIR))      # tự suy từ dataset
IMAGE_SIZE = 32        # CIFAR 32; CUB dùng 64 cho Phase 1 (CLIP/DINOv2 bỏ qua giá trị này)
MANY_THRESHOLD = 100   # CIFAR 100/20 ; CUB 15/6
FEW_THRESHOLD = 20
BUILD_DATASET = False

# ===== chung cho huấn luyện =====
PRETRAINED = False
BATCH_SIZE = 128
LEARNING_RATE = 0.1
EPOCHS = 200
SEED = 42
NUM_WORKERS = 2
DEVICE = None
VAL_FRACTION = 0.1
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None

# ===== Phase 1 (from-scratch) =====
METHODS = ["baseline", "balanced_softmax", "decoupling", "supcon", "meta", "cmo"]
CRT_EPOCHS = 10
CRT_LR = 0.1
PRETRAIN_EPOCHS = 200
PRETRAIN_LR = 0.5
TEMPERATURE = 0.07
N_WAY = 5
K_SHOT = 5
N_QUERY = 15
EPISODES_PER_EPOCH = 100
META_LR = 0.001
AUG_ALPHA = 1.0
MIX_PROB = 0.5

# ===== Phase 0 (reuse) =====
REUSE_METHODS = ["baseline", "balanced_softmax", "decoupling", "supcon"]
BEST_SINGLE = "cmo"
FUSION_WEIGHTS = {"many": 0.0, "medium": 0.3, "few": 0.8}
TAU_VALUES = [0.5, 0.75, 1.0]
CKPT_SOURCE = None   # Kaggle: "/kaggle/input/<notebook>/outputs" để nạp checkpoint Phase 1 từ session khác

# ===== Phase 2 / 3 (foundation, CLIP/DINOv2) =====
CLIP_MODEL = "ViT-B-32"
CLIP_PRETRAINED = "openai"
CLIP_PROMPT = "a photo of a {}"
LIFT_EPOCHS = 50
LIFT_LR = 1e-3
LIFT_BOTTLENECK = 64
TIPF_EPOCHS = 20
TIPF_LR = 1e-3
RUN_FT_ABLATION = False
FT_EPOCHS = 10
USE_LLM = True
USE_DINO = True
USE_DIFFUSION = True
USE_GLA = True
USE_CMO = True
USE_MIXUP = True
CMO_DIR = OUTPUT_DIR / "cmo"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
N_DESC_PER_CLASS = 8
DESC_JSON = OUTPUT_DIR / "class_descriptions.json"
DINO_MODEL = "dinov2_vits14"
DIFFUSION_STEPS = 100
DIFFUSION_EPOCHS = 200
MIXUP_ALPHA = 1.0
print("dataset:", DATA_DIR.name, "| classes:", NUM_CLASSES)

## 3. Định nghĩa 4 phase (mỗi phase một hàm, scope riêng)

In [ ]:
# PHASE 1 — train tất cả method từ đầu (run_all_methods)
def run_phase1():
    global DATA_DIR
    import random

    import numpy as np
    import torch

    from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_split,
                                        make_loader, split_indices_by_class, split_shot_groups,
                                        subset)
    from src.evaluation import visualize
    from src.evaluation.metrics import compute_metrics, format_metrics
    from src.trainers.engine import evaluate
    from src.utils.experiment import (compare_runs, create_run_dir, save_config,
                                       save_history, save_metrics)
    from src.utils.seed import resolve_device, set_seed

    device = resolve_device(DEVICE)
    print("Device:", device)

    if BUILD_DATASET:
        import subprocess
        subprocess.run([sys.executable, str(PROJECT_DIR / "data" / "prepare_datasets.py"),
                        "--dataset", "cifar100-lt",
                        "--data_dir", str(OUTPUT_DIR), "--overwrite"], check=True)
        DATA_DIR = OUTPUT_DIR / "CIFAR-100-LT"

    assert (DATA_DIR / "train").exists(), f"No train/ folder under {DATA_DIR}"
    class_counts = load_class_counts(DATA_DIR)
    shot_groups = split_shot_groups(class_counts)
    print("Classes per group:", {name: len(ids) for name, ids in shot_groups.items()})
    print("Head class count:", max(class_counts.values()), "| Tail class count:", min(class_counts.values()))

    # Load full train (augmented + eval views) and the test set.
    base_aug = load_split(DATA_DIR, "train", build_transforms(train=True, image_size=IMAGE_SIZE))
    base_eval = load_split(DATA_DIR, "train", build_transforms(train=False, image_size=IMAGE_SIZE))
    test_eval = load_split(DATA_DIR, "test", build_transforms(train=False, image_size=IMAGE_SIZE))

    # Optional smoke-test subsample (same indices for both train views, so they stay aligned).
    if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(base_aug.samples):
        keep = sorted(random.Random(SEED).sample(range(len(base_aug.samples)), MAX_TRAIN_SAMPLES))
        base_aug, base_eval = subset(base_aug, keep), subset(base_eval, keep)
    if MAX_TEST_SAMPLES is not None and MAX_TEST_SAMPLES < len(test_eval.samples):
        test_keep = sorted(random.Random(SEED).sample(range(len(test_eval.samples)), MAX_TEST_SAMPLES))
        test_eval = subset(test_eval, test_keep)

    # Stratified train/val split. val is used ONLY for model selection; test is never touched.
    train_idx, val_idx = split_indices_by_class([label for _, label in base_aug.samples], VAL_FRACTION, SEED)
    train_aug = subset(base_aug, train_idx)     # training (augmented)
    train_eval = subset(base_eval, train_idx)   # prototypes / t-SNE (eval transform)
    val_eval = subset(base_eval, val_idx)       # model selection (eval transform)

    train_loader = make_loader(train_aug, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    train_eval_loader = make_loader(train_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    val_loader = make_loader(val_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = make_loader(test_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print(f"train={len(train_aug)} | val={len(val_eval)} | test={len(test_eval)} images")

    # Look at the data once (long-tail profile).
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    visualize.plot_class_frequency(class_counts, OUTPUT_DIR)
    visualize.plot_shot_distribution(shot_groups, OUTPUT_DIR)

    def train_one(method, run_dir):
        """Dispatch one method. Returns (result_dict, history_df)."""
        if method == "baseline":
            from src.trainers.baseline_trainer import train_baseline
            model, history = train_baseline(train_loader, val_loader, NUM_CLASSES, device, run_dir,
                                            epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED)
            return evaluate(model, test_loader, device, collect_features=True), history

        if method == "balanced_softmax":
            from src.trainers.baseline_trainer import train_baseline
            from src.trainers.losses import BalancedSoftmaxLoss
            counts = [class_counts[c] for c in range(NUM_CLASSES)]
            model, history = train_baseline(train_loader, val_loader, NUM_CLASSES, device, run_dir,
                                            epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED,
                                            criterion=BalancedSoftmaxLoss(counts))
            return evaluate(model, test_loader, device, collect_features=True), history

        if method == "decoupling":
            from src.trainers.decoupling_trainer import train_decoupling
            model, history = train_decoupling(train_loader, train_aug, val_loader, NUM_CLASSES, device, run_dir,
                                              epochs=EPOCHS, learning_rate=LEARNING_RATE,
                                              crt_epochs=CRT_EPOCHS, crt_lr=CRT_LR,
                                              batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pretrained=PRETRAINED)
            return evaluate(model, test_loader, device, collect_features=True), history

        if method == "supcon":
            from src.trainers.contrastive_trainer import train_contrastive
            model, history, _ = train_contrastive(
                train_aug, val_loader, NUM_CLASSES, device, run_dir,
                pretrain_epochs=PRETRAIN_EPOCHS, probe_epochs=CRT_EPOCHS,
                batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                pretrain_lr=PRETRAIN_LR, probe_lr=CRT_LR,
                temperature=TEMPERATURE, pretrained=PRETRAINED, image_size=IMAGE_SIZE)
            return evaluate(model, test_loader, device, collect_features=True), history

        if method == "meta":
            from src.trainers.meta_trainer import evaluate_meta, train_meta
            encoder, history = train_meta(train_aug, val_eval, device, run_dir,
                                          epochs=EPOCHS, episodes_per_epoch=EPISODES_PER_EPOCH,
                                          n_way=N_WAY, k_shot=K_SHOT, n_query=N_QUERY,
                                          learning_rate=META_LR, pretrained=PRETRAINED, seed=SEED)
            return evaluate_meta(encoder, train_eval_loader, test_loader, NUM_CLASSES, device), history

        if method in ("mixup", "cutmix", "cmo"):
            from src.trainers.augment_trainer import train_augmented
            model, history = train_augmented(train_aug, val_loader, NUM_CLASSES, device, run_dir, class_counts,
                                             mode=method, alpha=AUG_ALPHA, mix_prob=MIX_PROB,
                                             epochs=EPOCHS, learning_rate=LEARNING_RATE,
                                             batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                                             pretrained=PRETRAINED, use_balanced_softmax=True)
            return evaluate(model, test_loader, device, collect_features=True), history

        raise ValueError(f"Unknown method: {method!r}")


    histories = {}        # method -> per-epoch history (for overlay curves)
    all_results = {}      # method -> result dict (for confusion matrices / t-SNE)

    for method in METHODS:
        print(f"\n========== {method} ==========")
        set_seed(SEED)                                   # fair, reproducible start per method
        run_dir = create_run_dir(OUTPUT_DIR, run_name=method)
        result, history = train_one(method, run_dir)

        metrics = compute_metrics(result["y_true"], result["y_pred"], NUM_CLASSES, shot_groups)
        print(format_metrics(metrics))

        save_config(run_dir, {"method": method, "epochs": EPOCHS, "pretrained": PRETRAINED,
                              "image_size": IMAGE_SIZE, "seed": SEED, "batch_size": BATCH_SIZE})
        save_metrics(run_dir, metrics)
        save_history(run_dir, history)
        # Per-method figures (also saved in each run folder).
        visualize.plot_confusion_matrices(result["y_true"], result["y_pred"], NUM_CLASSES, run_dir)
        if "features" in result:
            visualize.plot_tsne(result["features"], result["y_true"], run_dir)

        histories[method] = history
        all_results[method] = result

    print("\nAll methods finished.")

    comparison = compare_runs(OUTPUT_DIR)
    comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
    comparison

    # 1) Grouped bar chart across metrics (all methods in one figure)
    visualize.plot_metric_comparison(comparison, OUTPUT_DIR)

    # 2) One overlay figure per training measure (all comparable methods as lines)
    curve_histories = {m: h for m, h in histories.items() if m != "meta"}
    overlay_paths = visualize.plot_curves_overlay(curve_histories, OUTPUT_DIR)

    print("Saved to", OUTPUT_DIR)
    print(sorted(p.name for p in OUTPUT_DIR.glob("*.png")))

    # Show the figures inline
    from IPython.display import Image, display

    for name in ["comparison_metrics.png", "overlay_val_accuracy.png", "overlay_val_loss.png"]:
        path = OUTPUT_DIR / name
        if path.exists():
            print(name)
            display(Image(filename=str(path)))

In [ ]:
# PHASE 0 — tái dùng checkpoint: ensemble / tier_fusion / tau-norm / CLIP fusion
def run_phase0():
    import numpy as np
    import torch

    from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_split,
                                        make_loader, split_indices_by_class, split_shot_groups,
                                        subset)
    from src.evaluation import visualize
    from src.evaluation.ensemble import ensemble_predict, load_classifier, predict_probs
    from src.evaluation.metrics import compute_metrics, format_metrics
    from src.evaluation.posthoc import tau_normalized_predict, tier_fusion
    from src.utils.experiment import compare_runs, create_run_dir, save_metrics
    from src.utils.seed import resolve_device, set_seed

    set_seed(42)
    device = resolve_device(DEVICE)

    class_counts = load_class_counts(DATA_DIR)
    shot_groups = split_shot_groups(class_counts)
    base_eval = load_split(DATA_DIR, "train", build_transforms(train=False, image_size=IMAGE_SIZE))
    train_idx, val_idx = split_indices_by_class([l for _, l in base_eval.samples], VAL_FRACTION, SEED)
    train_eval = subset(base_eval, train_idx)   # training split (for prototypes)
    val_eval = subset(base_eval, val_idx)       # validation (for tau selection)
    test_eval = load_split(DATA_DIR, "test", build_transforms(train=False, image_size=IMAGE_SIZE))
    train_eval_loader = make_loader(train_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    val_loader = make_loader(val_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = make_loader(test_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print("device:", device, "| reusing:", REUSE_METHODS)


    def record(name, result):
        """Compute + save metrics for a reuse technique, and print the key numbers."""
        metrics = compute_metrics(result["y_true"], result["y_pred"], NUM_CLASSES, shot_groups)
        run_dir = create_run_dir(OUTPUT_DIR, name)
        save_metrics(run_dir, metrics)
        print(f"\n=== {name} ===")
        print(format_metrics(metrics))
        return metrics

    # Bring checkpoints from a previous session into OUTPUT_DIR (no-op if CKPT_SOURCE is None).
    import shutil

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if CKPT_SOURCE is not None:
        source = Path(CKPT_SOURCE)
        assert source.exists(), f"CKPT_SOURCE not found: {source}"
        for run in sorted(p for p in source.iterdir() if p.is_dir()):
            dst = OUTPUT_DIR / run.name
            if not dst.exists():
                shutil.copytree(run, dst)
        print("imported runs:", sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_dir()))
    else:
        print("CKPT_SOURCE=None -> using checkpoints already in", OUTPUT_DIR)

    models = {}
    for method in REUSE_METHODS:
        run_dir = OUTPUT_DIR / method
        if (run_dir / "best_model.pt").exists():
            models[method] = load_classifier(run_dir, NUM_CLASSES, device, pretrained=PRETRAINED)
            print("loaded", method)
        else:
            print("skip (no checkpoint):", method)

    ens = list(models.values())
    record("ensemble", ensemble_predict(ens, test_loader, device, tta=False))
    record("ensemble_tta", ensemble_predict(ens, test_loader, device, tta=True))

    best = models[BEST_SINGLE]
    record("tier_fusion", tier_fusion(best, train_eval_loader, test_loader,
                                      NUM_CLASSES, shot_groups, device, weights=FUSION_WEIGHTS))

    # Select tau on the validation split, then report the chosen tau on test.
    val_score = {}
    for tau in TAU_VALUES:
        r = tau_normalized_predict(best, val_loader, device, tau=tau)
        val_score[tau] = compute_metrics(r["y_true"], r["y_pred"], NUM_CLASSES, shot_groups)["balanced_accuracy"]
    best_tau = max(val_score, key=val_score.get)
    print("val balanced_acc by tau:", {k: round(v, 4) for k, v in val_score.items()}, "-> best tau =", best_tau)
    record(f"tau_norm", tau_normalized_predict(best, test_loader, device, tau=best_tau))

    import numpy as np
    from src.evaluation.metrics import balanced_accuracy
    from src.experts.clip_expert import CIFAR100_CLASSES, clip_probs, load_clip

    best_vision = models[BEST_SINGLE]   # set BEST_SINGLE='cmo' for the strongest vision model
    clip_bundle = load_clip(CIFAR100_CLASSES, device, model_name='ViT-B-32', pretrained='openai')

    # Per-sample probabilities on val (to tune alpha) and test (to report).
    pv_val, yv = predict_probs(best_vision, val_loader, device)
    pc_val, yv2 = clip_probs(val_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
    pv_test, yt = predict_probs(best_vision, test_loader, device)
    pc_test, yt2 = clip_probs(test_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
    assert (yv == yv2).all() and (yt == yt2).all(), 'sample order mismatch between experts'

    # Single experts for reference.
    record('vision_only', {'y_true': yt, 'y_pred': pv_test.argmax(1)})
    record('clip_only', {'y_true': yt, 'y_pred': pc_test.argmax(1)})

    # Tune the CLIP weight alpha on validation (balanced accuracy), then apply on test.
    alphas = np.linspace(0.0, 1.0, 21)
    best_alpha = max(alphas, key=lambda a: balanced_accuracy(yv, (a * pc_val + (1 - a) * pv_val).argmax(1)))
    print('best CLIP weight alpha (on val):', round(float(best_alpha), 2))
    fused = best_alpha * pc_test + (1 - best_alpha) * pv_test
    record('vlm_fusion', {'y_true': yt, 'y_pred': fused.argmax(1)})

    comparison = compare_runs(OUTPUT_DIR)
    comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
    visualize.plot_metric_comparison(comparison, OUTPUT_DIR)
    comparison

In [ ]:
# PHASE 2 — adapt CLIP đóng băng: Tip-Adapter / LIFT / (FT-ablation)
def run_phase2():
    global NUM_CLASSES
    import numpy as np
    import pandas as pd
    import torch

    from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_class_names, load_split,
                                        split_indices_by_class, split_shot_groups, subset)
    from src.evaluation import visualize
    from src.evaluation.metrics import compute_metrics, format_metrics
    from src.experts.clip_expert import (clip_zero_shot_logits,
                                         encode_clip_features, load_clip)
    from src.experts import tip_adapter as ta
    from src.experts import lift as lift_mod
    from src.utils.experiment import compare_runs, create_run_dir, save_metrics
    from src.utils.seed import resolve_device, set_seed

    set_seed(SEED)
    device = resolve_device(DEVICE)

    # class_counts.json is the full LT profile; for the loss we use the actual train-subset counts.
    class_counts_full = load_class_counts(DATA_DIR)
    NUM_CLASSES = len(class_counts_full)        # auto-match the dataset (100 CIFAR-100, 200 CUB)
    CLASS_NAMES = load_class_names(DATA_DIR)    # readable names for CLIP prompts
    shot_groups = split_shot_groups(class_counts_full, MANY_THRESHOLD, FEW_THRESHOLD)

    eval_tf = build_transforms(train=False, image_size=32)  # placeholder; CLIP swaps in its own preprocess
    base_train = load_split(DATA_DIR, "train", eval_tf)
    train_idx, val_idx = split_indices_by_class([l for _, l in base_train.samples], VAL_FRACTION, SEED)
    train_eval = subset(base_train, train_idx)
    val_eval = subset(base_train, val_idx)
    test_eval = load_split(DATA_DIR, "test", eval_tf)
    print("device:", device, "| train", len(train_eval), "| val", len(val_eval), "| test", len(test_eval))


    def record(name, result):
        """Compute + save metrics for one method, print the headline numbers, return them."""
        metrics = compute_metrics(result["y_true"], result["y_pred"], NUM_CLASSES, shot_groups)
        run_dir = create_run_dir(OUTPUT_DIR, name)
        save_metrics(run_dir, metrics)
        print(f"\n=== {name} ===")
        print(format_metrics(metrics))
        return metrics

    clip_bundle = load_clip(CLASS_NAMES, device, model_name=CLIP_MODEL,
                            pretrained=CLIP_PRETRAINED, prompt=CLIP_PROMPT)

    f_train, y_train = encode_clip_features(train_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS, multi_gpu=USE_MULTI_GPU)
    f_val, y_val = encode_clip_features(val_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS, multi_gpu=USE_MULTI_GPU)
    f_test, y_test = encode_clip_features(test_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS, multi_gpu=USE_MULTI_GPU)

    # Smoke test: keep a RANDOM subset (features are ordered by class, so [:N] would keep only
    # the first few head classes — random keeps the long-tail shape across all 100 classes).
    if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(y_train):
        keep = torch.randperm(len(y_train), generator=torch.Generator().manual_seed(SEED))[:MAX_TRAIN_SAMPLES]
        f_train, y_train = f_train[keep], y_train[keep]

    # Per-class training counts AFTER the val split (the distribution the loss should correct).
    train_counts = list(np.bincount(y_train.numpy(), minlength=NUM_CLASSES))

    # Zero-shot CLIP logits for each split (cosine sim to text prototypes, on CLIP's scale).
    clip_train = clip_zero_shot_logits(f_train, clip_bundle)
    clip_val = clip_zero_shot_logits(f_val, clip_bundle)
    clip_test = clip_zero_shot_logits(f_test, clip_bundle)

    record("clip_only", {"y_true": y_test.numpy(), "y_pred": clip_test.argmax(1).numpy()})
    print("feature dim:", f_train.shape[1])

    keys, values = ta.build_cache(f_train, y_train, NUM_CLASSES)

    alpha, beta, val_acc = ta.tune_alpha_beta(f_val, keys, values, clip_val, y_val.numpy())
    print(f"tuned on val -> alpha={alpha:.2f}, beta={beta:.2f}, val bal-acc={val_acc:.4f}")
    record("tip_adapter", ta.predict(f_test, y_test, keys, values, clip_test, alpha, beta))

    tipf = ta.TipAdapterF(keys, values, beta)
    ta.train_tip_adapter_f(tipf, f_train, clip_train, y_train, alpha, train_counts, device,
                           epochs=TIPF_EPOCHS, lr=TIPF_LR)
    record("tip_adapter_f",
           ta.predict(f_test, y_test, keys, values, clip_test, alpha, beta, model=tipf, device=device))

    lift_model, info = lift_mod.train_lift(
        f_train, y_train, f_val, y_val,
        clip_bundle["text_features"].float(), clip_bundle["logit_scale"],
        train_counts, NUM_CLASSES, device,
        epochs=LIFT_EPOCHS, lr=LIFT_LR, bottleneck=LIFT_BOTTLENECK)
    print("best val balanced-acc:", round(info["best_val_balanced_accuracy"], 4))

    run_dir = create_run_dir(OUTPUT_DIR, "lift")
    pd.DataFrame(info["history"]).to_csv(run_dir / "metrics.csv", index=False)
    record("lift", lift_mod.predict(lift_model, f_test, y_test, device))

    if RUN_FT_ABLATION:
        import numpy as np
        from src.experts import clip_finetune
        feat_dim = f_train.shape[1]
        ft_counts = list(np.bincount([l for _, l in train_eval.samples], minlength=NUM_CLASSES))
        lr_by_depth = {"linear_probe": 1e-3, "last_block": 1e-4, "full_ft": 1e-5}
        print("fine-tuning-depth ablation (heavy; trains the ViT on images):")
        for depth in clip_finetune.DEPTHS:
            net, info = clip_finetune.train_clip_finetune(
                clip_bundle["model"], clip_bundle["preprocess"], train_eval, val_eval,
                NUM_CLASSES, ft_counts, feat_dim, device, depth,
                epochs=FT_EPOCHS, lr=lr_by_depth[depth], batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
            print(f"  {depth:13s} {info['trainable_params']:>10,} params  val bal={info['best_val_balanced_accuracy']:.4f}")
            record(f"ft_{depth}", clip_finetune.predict(net, test_eval, clip_bundle["preprocess"], device, BATCH_SIZE, NUM_WORKERS))
    else:
        print("RUN_FT_ABLATION=False -> skipping the heavy fine-tuning ablation")

    vlm_methods = ["clip_only", "tip_adapter", "tip_adapter_f", "lift", "vlm_fusion",
                   "ft_linear_probe", "ft_last_block", "ft_full_ft"]
    comparison = compare_runs(OUTPUT_DIR)
    comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)

    vlm_table = comparison[comparison["method"].isin(vlm_methods)].reset_index(drop=True)
    vlm_table.to_csv(OUTPUT_DIR / "comparison_vlm.csv", index=False)
    visualize.plot_metric_comparison(comparison, OUTPUT_DIR)
    vlm_table

In [ ]:
# PHASE 3 — nghiên cứu nguồn tri thức: LLM / DINOv2 / diffusion / mixup / GLA / fusion
def run_phase3():
    global NUM_CLASSES
    import numpy as np
    import pandas as pd
    import torch

    from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_class_names, load_split, make_loader,
                                        split_indices_by_class, split_shot_groups, subset)
    from src.evaluation import visualize
    from src.evaluation.metrics import compute_metrics, format_metrics
    from src.evaluation import fusion
    from src.experts.clip_expert import (clip_zero_shot_logits, encode_clip_features,
                                         encode_text_prototypes, load_clip)
    from src.experts import gla, lift as lift_mod
    from src.utils.experiment import compare_runs, create_run_dir, save_metrics
    from src.utils.seed import resolve_device, set_seed

    set_seed(SEED)
    device = resolve_device(DEVICE)
    class_counts = load_class_counts(DATA_DIR)
    NUM_CLASSES = len(class_counts)            # auto-match the dataset (100 CIFAR-100, 200 CUB)
    CLASS_NAMES = load_class_names(DATA_DIR)   # readable names for CLIP / LLM prompts
    shot_groups = split_shot_groups(class_counts, MANY_THRESHOLD, FEW_THRESHOLD)

    eval_tf = build_transforms(train=False, image_size=32)
    base_train = load_split(DATA_DIR, "train", eval_tf)
    train_idx, val_idx = split_indices_by_class([l for _, l in base_train.samples], VAL_FRACTION, SEED)
    train_eval, val_eval = subset(base_train, train_idx), subset(base_train, val_idx)
    test_eval = load_split(DATA_DIR, "test", eval_tf)

    clip_bundle = load_clip(CLASS_NAMES, device, model_name=CLIP_MODEL, pretrained=CLIP_PRETRAINED)
    f_train, y_train = encode_clip_features(train_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS, multi_gpu=USE_MULTI_GPU)
    f_val, y_val = encode_clip_features(val_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS, multi_gpu=USE_MULTI_GPU)
    f_test, y_test = encode_clip_features(test_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS, multi_gpu=USE_MULTI_GPU)
    if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(y_train):
        keep = torch.randperm(len(y_train), generator=torch.Generator().manual_seed(SEED))[:MAX_TRAIN_SAMPLES]
        f_train, y_train = f_train[keep], y_train[keep]
    train_counts = list(np.bincount(y_train.numpy(), minlength=NUM_CLASSES))
    yv, yt = y_val.numpy(), y_test.numpy()

    # Expert registry: name -> per-sample logits/probs on val and test (aligned to yv / yt).
    EXP_VAL, EXP_TEST = {}, {}

    def add_expert(name, val_scores, test_scores):
        EXP_VAL[name] = np.asarray(val_scores)
        EXP_TEST[name] = np.asarray(test_scores)
        record(name, EXP_TEST[name].argmax(1))

    def record(name, y_pred):
        m = compute_metrics(yt, y_pred, NUM_CLASSES, shot_groups)
        save_metrics(create_run_dir(OUTPUT_DIR, name), m)
        print(f"{name:18s} bal={m['balanced_accuracy']:.4f}  many={m['many_shot_accuracy']:.3f}  "
              f"med={m['medium_shot_accuracy']:.3f}  few={m['few_shot_accuracy']:.3f}")
        return m

    print("features:", f_train.shape, "| device:", device)

    clip_val = clip_zero_shot_logits(f_val, clip_bundle).numpy()
    clip_test = clip_zero_shot_logits(f_test, clip_bundle).numpy()
    add_expert("clip_zeroshot", clip_val, clip_test)
    proto_init = clip_bundle["text_features"].float()   # LIFT cosine-head init (may be replaced by A)

    if USE_LLM:
        from src.experts import llm_prompts
        if DESC_JSON.exists():
            descriptions = llm_prompts.load_descriptions(DESC_JSON)
            print("loaded cached descriptions:", len(descriptions), "classes")
        else:
            print("generating descriptions with", LLM_MODEL, "(runs once)...")
            descriptions = llm_prompts.generate_descriptions(CLASS_NAMES, device,
                                                             llm_model_name=LLM_MODEL, n_per_class=N_DESC_PER_CLASS)
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            llm_prompts.save_descriptions(descriptions, DESC_JSON)
        print("example [leopard]:", descriptions.get("leopard", ["-"])[0][:90])

        prompts = llm_prompts.prompts_in_label_order(descriptions, CLASS_NAMES)
        proto_llm = encode_text_prototypes(clip_bundle, prompts, device)
        add_expert("clip_llm", (clip_bundle["logit_scale"] * f_val @ proto_llm.cpu().t()).numpy(),
                               (clip_bundle["logit_scale"] * f_test @ proto_llm.cpu().t()).numpy())
        proto_init = proto_llm.float()   # use the richer prototypes to seed LIFT

    def lift_expert(feats_tr, lab_tr, init, name, **kw):
        model, info = lift_mod.train_lift(feats_tr, lab_tr, f_val, y_val, init.to(device),
                                          clip_bundle["logit_scale"], list(np.bincount(lab_tr.numpy(), minlength=NUM_CLASSES)),
                                          NUM_CLASSES, device, epochs=LIFT_EPOCHS, lr=LIFT_LR, **kw)
        with torch.no_grad():
            vl = model(f_val.to(device)).cpu().numpy()
            tl = model(f_test.to(device)).cpu().numpy()
        print(f"  [{name}] best val bal={info['best_val_balanced_accuracy']:.4f}")
        return vl, tl

    vl, tl = lift_expert(f_train, y_train, proto_init, "lift_clip")
    add_expert("lift_clip", vl, tl)

    if USE_DIFFUSION:
        from src.experts import feature_diffusion as fd
        diff = fd.FeatureDiffusion(f_train.shape[1], NUM_CLASSES, device, num_steps=DIFFUSION_STEPS)
        diff.train(f_train, y_train, epochs=DIFFUSION_EPOCHS)
        f_aug, y_aug = fd.augment_tail(f_train, y_train, diff, NUM_CLASSES)
        print(f"  augmented {len(y_train)} -> {len(y_aug)} features")
        vl, tl = lift_expert(f_aug, y_aug, proto_init, "lift_clip_diff")
        add_expert("lift_clip_diff", vl, tl)

    if USE_MIXUP:
        vl, tl = lift_expert(f_train, y_train, proto_init, "lift_clip_mixup", mixup_alpha=MIXUP_ALPHA)
        add_expert("lift_clip_mixup", vl, tl)

    if USE_DINO:
        from src.experts import dino_expert
        dino_bundle = dino_expert.load_dino(device, model_name=DINO_MODEL)
        d_train, _ = dino_expert.encode_dino_features(train_eval, dino_bundle, device, BATCH_SIZE, NUM_WORKERS, multi_gpu=USE_MULTI_GPU)
        d_val, _ = dino_expert.encode_dino_features(val_eval, dino_bundle, device, BATCH_SIZE, NUM_WORKERS, multi_gpu=USE_MULTI_GPU)
        d_test, _ = dino_expert.encode_dino_features(test_eval, dino_bundle, device, BATCH_SIZE, NUM_WORKERS, multi_gpu=USE_MULTI_GPU)
        if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(d_train):
            d_train = d_train[keep]
        proto_dino = dino_expert.class_mean_prototypes(d_train, y_train, NUM_CLASSES)
        dm, dinfo = lift_mod.train_lift(d_train, y_train, d_val, y_val, proto_dino.to(device), 30.0,
                                        train_counts, NUM_CLASSES, device, epochs=LIFT_EPOCHS, lr=LIFT_LR)
        with torch.no_grad():
            add_expert("dino_lift", dm(d_val.to(device)).cpu().numpy(), dm(d_test.to(device)).cpu().numpy())

    if USE_DINO:
        import copy
        import torch.nn as nn
        from src.evaluation.metrics import balanced_accuracy

        def linear_probe(ftr, ytr, fva, yva, fte, num_classes, device, epochs, lr=1e-3, criterion=None):
            head = nn.Linear(ftr.shape[1], num_classes).to(device)
            opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-2)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
            crit = criterion if criterion is not None else nn.CrossEntropyLoss()  # default: plain CE (raw-backbone baseline)
            loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(ftr, ytr.long()),
                                                 batch_size=256, shuffle=True)
            best_state, best = copy.deepcopy(head.state_dict()), -1.0
            for _ in range(epochs):
                head.train()
                for xb, yb in loader:
                    loss = crit(head(xb.to(device)), yb.to(device))
                    opt.zero_grad(); loss.backward(); opt.step()
                sched.step()
                head.eval()
                with torch.no_grad():
                    v = balanced_accuracy(yva.numpy(), head(fva.to(device)).argmax(1).cpu().numpy())
                if v > best:
                    best, best_state = v, copy.deepcopy(head.state_dict())
            head.load_state_dict(best_state); head.eval()
            with torch.no_grad():
                return head(fte.to(device)).cpu().numpy()

        tl_lp = linear_probe(d_train, y_train, d_val, y_val, d_test, NUM_CLASSES, device, LIFT_EPOCHS)
        record("dino_linear_probe", tl_lp.argmax(1))   # table only; not added to fusion

    if USE_DINO:
        from src.trainers.losses import BalancedSoftmaxLoss
        tl_bs = linear_probe(d_train, y_train, d_val, y_val, d_test, NUM_CLASSES, device,
                             LIFT_EPOCHS, criterion=BalancedSoftmaxLoss(train_counts).to(device))
        record("dino_linear_probe_bs", tl_bs.argmax(1))   # table only; not added to fusion

    if USE_CMO and (CMO_DIR / "best_model.pt").exists():
        import torch as _t
        from src.evaluation.ensemble import load_classifier, predict_probs
        # Auto-detect how cmo was trained from its conv1 stem (7x7 = ImageNet/224, 3x3 = CIFAR/32)
        # so we load the matching architecture AND evaluate at the right resolution.
        _state = _t.load(CMO_DIR / "best_model.pt", map_location="cpu", weights_only=False)
        _pretrained = _state["model_state_dict"]["encoder.conv1.weight"].shape[-1] == 7
        _img = 224 if _pretrained else 32
        print(f"cmo control: pretrained={_pretrained}, image_size={_img}")
        _tf = build_transforms(train=False, image_size=_img)
        _cval = subset(load_split(DATA_DIR, "train", _tf), val_idx)
        _ctest = load_split(DATA_DIR, "test", _tf)
        cmo = load_classifier(CMO_DIR, NUM_CLASSES, device, pretrained=_pretrained)
        pv, yv_c = predict_probs(cmo, make_loader(_cval, BATCH_SIZE, False, NUM_WORKERS), device)
        pt, yt_c = predict_probs(cmo, make_loader(_ctest, BATCH_SIZE, False, NUM_WORKERS), device)
        assert (yv_c == yv).all() and (yt_c == yt).all(), "cmo sample order mismatch"
        add_expert("cmo", pv, pt)          # stored as probabilities; fusion.to_probs handles it
    else:
        print("cmo checkpoint not found -> skipping control (set CMO_DIR / USE_CMO=False)")

    if USE_GLA:
        import torch as _t
        print("GLA debiasing (before -> after, balanced acc):")
        for name in [n for n in ("clip_zeroshot", "clip_llm", "lift_clip", "dino_lift") if n in EXP_TEST]:
            vlog, tlog = _t.tensor(EXP_VAL[name]), _t.tensor(EXP_TEST[name])
            bias = gla.estimate_log_bias(tlog)
            s, _ = gla.tune_strength(vlog, yv, gla.estimate_log_bias(vlog))
            before = compute_metrics(yt, tlog.argmax(1).numpy(), NUM_CLASSES, shot_groups)["balanced_accuracy"]
            after = compute_metrics(yt, gla.gla_adjust(tlog, bias, s).argmax(1).numpy(), NUM_CLASSES, shot_groups)["balanced_accuracy"]
            print(f"  {name:16s} {before:.4f} -> {after:.4f}  (strength={s:.2f})")
            # keep the debiased scores for fusion
            EXP_VAL[name] = gla.gla_adjust(vlog, gla.estimate_log_bias(vlog), s).numpy()
            EXP_TEST[name] = gla.gla_adjust(tlog, bias, s).numpy()

    val_probs = {k: fusion.to_probs(v) for k, v in EXP_VAL.items()}
    test_probs = {k: fusion.to_probs(v) for k, v in EXP_TEST.items()}
    names = list(test_probs.keys())
    print("experts in fusion:", names)

    group_w, val_bal = fusion.tune_group_weights([val_probs[n] for n in names], yv, shot_groups, NUM_CLASSES)
    class_group = np.empty(NUM_CLASSES, dtype=object)
    for g, ids in shot_groups.items():
        for c in ids:
            class_group[c] = g
    fused_test = fusion.fuse_group([test_probs[n] for n in names], group_w, class_group)
    add_expert("fusion_tailaware", fused_test, fused_test)   # logits==probs here; argmax identical

    print("\nper-shot-group weights (over", names, "):")
    for g in ("many", "medium", "few"):
        print(f"  {g:7s}", tuple(round(w, 2) for w in group_w[g]))

    rep = fusion.complementarity_report(test_probs, yt, NUM_CLASSES, shot_groups, fused_test)
    rep_df = pd.DataFrame(rep).T[["balanced_accuracy", "many_shot_accuracy", "medium_shot_accuracy", "few_shot_accuracy"]]
    rep_df = rep_df.sort_values("balanced_accuracy", ascending=False)
    rep_df.to_csv(OUTPUT_DIR / "knowledge_sources.csv")
    rep_df.round(4)

    # Greedy fusion: monotone-improving over the best single per group (Caruana 2004).
    gw_greedy, valbal_greedy = fusion.greedy_group_weights(
        [val_probs[n] for n in names], yv, shot_groups, NUM_CLASSES)
    fused_greedy = fusion.fuse_group([test_probs[n] for n in names], gw_greedy, class_group)
    record("fusion_greedy", fused_greedy.argmax(1))   # table only; not re-fed into any fusion
    print(f"greedy val bal={valbal_greedy:.4f} | per-group weights (over {names}):")
    for g in ("many", "medium", "few"):
        print(f"  {g:7s}", tuple(round(w, 2) for w in gw_greedy[g]))

    comparison = compare_runs(OUTPUT_DIR)
    comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
    vlm = ["clip_zeroshot", "clip_llm", "lift_clip", "lift_clip_diff", "lift_clip_mixup", "dino_lift", "dino_linear_probe", "dino_linear_probe_bs", "fusion_tailaware", "fusion_greedy", "cmo"]
    comparison[comparison["method"].isin(vlm)].reset_index(drop=True)

## 4. Chạy các phase theo toggle (thứ tự 1 → 0 → 2 → 3)

In [ ]:
import torch
print("GPUs:", torch.cuda.device_count())
if RUN_PHASE1: print("\n########## PHASE 1 ##########"); run_phase1()
if RUN_PHASE0: print("\n########## PHASE 0 ##########"); run_phase0()
if RUN_PHASE2: print("\n########## PHASE 2 ##########"); run_phase2()
if RUN_PHASE3: print("\n########## PHASE 3 ##########"); run_phase3()

## 5. Bảng tổng hợp cuối

In [ ]:
from src.utils.experiment import compare_runs
comparison = compare_runs(OUTPUT_DIR)
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
comparison